In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd

# Membuat Spark Session
spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Membuat DataFrame referensi cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"]
}

df_cabang = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

# Membaca data transaksi dari HDFS
df_transaksi = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv")

# Membuat kolom pendapatan
from pyspark.sql.functions import expr

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    expr("unit_terjual * harga_satuan")
)

# Menampilkan data
print("Data Cabang")
df_cabang.show()

print("Data Transaksi")
df_transaksi.show(5, truncate=False)

print("Jumlah transaksi:", df_transaksi.count())

Data Cabang
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+

Data Transaksi
+--------+----------------------+----------+------------+------------+----------+
|order_id|kategori              |kota      |unit_terjual|harga_satuan|pendapatan|
+--------+----------------------+----------+------------+------------+----------+
|TRX-0   |Makanan & Minuman     |Purworejo |8           |75000       |600000    |
|TRX-1   |Elektronik            |Solo      |9           |75000       |675000    |
|TRX-2   |Kesehatan & Kecantikan|Solo      |6           |100000      |600000    |
|TRX-3   |Fashion               |Yogyakarta|6           |100000      |600000    |
|TRX-4   |Elektronik            |Yogyakarta|2  

A. Join dan perbandingan Target

In [28]:
from pyspark.sql.functions import sum as spark_sum, col

hasil = (
    df_transaksi
    .groupBy("kota")
    .agg(
        spark_sum("pendapatan").alias("total_pendapatan")
    )
    .join(
        df_cabang,
        on="kota",
        how="inner"
    )
    .withColumn(
        "pencapaian_persen",
        (col("total_pendapatan") / col("target_bulanan")) * 100
    )
    .orderBy(
        col("pencapaian_persen").desc()
    )
)

hasil.show(truncate=False)

+----------+----------------+--------------+----------+------------------+
|kota      |total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen |
+----------+----------------+--------------+----------+------------------+
|Purworejo |45650000        |30000000      |Fitri     |152.16666666666669|
|Solo      |33475000        |40000000      |Bayu      |83.6875           |
|Yogyakarta|47275000        |60000000      |Joko      |78.79166666666667 |
|Magelang  |31650000        |45000000      |Rani      |70.33333333333334 |
|Semarang  |38175000        |55000000      |Sari      |69.4090909090909  |
+----------+----------------+--------------+----------+------------------+



B. Window Function

In [30]:
from pyspark.sql.functions import sum as spark_sum, col, row_number
from pyspark.sql.window import Window

# Hitung total pendapatan berdasarkan kota dan kategori
total_kategori = (
    df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
)
# Membuat window berdasarkan kota
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Memberikan nomor urut pada setiap kategori
hasil = (
    total_kategori.withColumn("peringkat",row_number().over(window_kota)).filter(col("peringkat") == 1)
)
hasil.show(truncate=False)

+----------+----------------------+----------------+---------+
|kota      |kategori              |total_pendapatan|peringkat|
+----------+----------------------+----------------+---------+
|Magelang  |Kesehatan & Kecantikan|7275000         |1        |
|Purworejo |Kesehatan & Kecantikan|10075000        |1        |
|Semarang  |Rumah Tangga          |11125000        |1        |
|Solo      |Kesehatan & Kecantikan|8425000         |1        |
|Yogyakarta|Fashion               |13325000        |1        |
+----------+----------------------+----------------+---------+



C. Spark SQL

In [31]:
df_transaksi.createOrReplaceTempView("transaksi")
df_cabang.createOrReplaceTempView("target")

hasil = spark.sql("""
    SELECT
        t.kota,
        g.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target g
        ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil.show(truncate=False)

+----------+----------+----------------+
|kota      |pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
|Purworejo |Fitri     |116             |
|Yogyakarta|Joko      |110             |
|Solo      |Bayu      |95              |
|Semarang  |Sari      |93              |
|Magelang  |Rani      |86              |
+----------+----------+----------------+



D. Kesimpulan

Berdasarkan hasil analisis pada bagian A, cabang dengan hasil pencapaian target paling tinggi adalah Purworejo, dengan total pendapatan sebesar Rp45.650.000 dan pencapaian target sebesar 152,1%. Artinya, pendapatan Purworejo sudah melebihi target bulanan sebesar Rp30.000.000. Sementara itu, cabang yang perlu mendapat perhatian lebih adalah Semarang, karena total pendapatannya hanya Rp38.175.000 dengan pencapaian sekitar 63%, sehingga masih berada di bawah target bulanan.

Pada bagian B, dapat diketahui juga kategori dengan pendapatan paling tinggi di setiap kota. Sebagai contoh, di Yogyakarta, kategori Fashion memiliki pendapatan tertinggi sebesar Rp13.325.000. Data ini dapat membantu melihat kategori produk yang memberikan pendapatan terbesar di masing-masing cabang.

Secara keseluruhan, hasil analisis menggunakan join dan window function dapat membantu melihat pencapaian pendapatan terhadap target serta mengetahui kategori yang memiliki pendapatan terbesar. Hasil tersebut dapat digunakan sebagai bahan evaluasi dan pertimbangan dalam menyusun strategi penjualan di setiap cabang.

EXPLORASI

In [32]:
# MELIHAT STRUKTUR DAN ISI DATA
df_transaksi.printSchema()
df_transaksi.show(10, truncate=False)
print("Jumlah transaksi:", df_transaksi.count())

root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+----------------------+----------+------------+------------+----------+
|order_id|kategori              |kota      |unit_terjual|harga_satuan|pendapatan|
+--------+----------------------+----------+------------+------------+----------+
|TRX-0   |Makanan & Minuman     |Purworejo |8           |75000       |600000    |
|TRX-1   |Elektronik            |Solo      |9           |75000       |675000    |
|TRX-2   |Kesehatan & Kecantikan|Solo      |6           |100000      |600000    |
|TRX-3   |Fashion               |Yogyakarta|6           |100000      |600000    |
|TRX-4   |Elektronik            |Yogyakarta|2           |75000       |150000    |
|TRX-5   |Kesehatan & Kecantikan|Magelang  |4           |100000      |400000    |
|TRX-6 

In [37]:
#mengelompokan total pendapatan berdasarkan kota dan kategori
df_transaksi.groupBy("kota").pivot("kategori").sum("pendapatan").show()

+----------+----------+--------+----------------------+-----------------+------------+
|      kota|Elektronik| Fashion|Kesehatan & Kecantikan|Makanan & Minuman|Rumah Tangga|
+----------+----------+--------+----------------------+-----------------+------------+
|  Magelang|   7075000| 7200000|               7275000|          5225000|     4875000|
|  Semarang|   4500000| 4875000|               8475000|          9200000|    11125000|
|      Solo|   7350000| 4350000|               8425000|          7750000|     5600000|
| Purworejo|   9500000| 7575000|              10075000|          9600000|     8900000|
|Yogyakarta|  10500000|13325000|               7375000|          9200000|     6875000|
+----------+----------+--------+----------------------+-----------------+------------+



Tutup Sparksession

In [38]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
